# Mental Health Agentic AI Platform
## Data Exploration & Model Prototyping
---

### 1. Setup & Imports

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Setup complete')

### 2. Emotion Classifier Quick Test

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model='j-hartmann/emotion-english-distilroberta-base',
    top_k=None
)

test_texts = [
    'I feel so happy and excited today!',
    'I am really anxious about everything.',
    'Everything feels hopeless and dark.',
    'I am furious about what happened.',
    'I feel perfectly fine, nothing special.'
]

for text in test_texts:
    result = classifier(text)[0]
    top = max(result, key=lambda x: x['score'])
    print(f'Text: {text[:50]}')
    print(f'  -> {top["label"]} ({top["score"]:.2%})')
    print()

### 3. Embedding Model Test

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    'I feel anxious and cannot calm down',
    'Try the 4-7-8 breathing technique for anxiety',
    'The weather is nice today'
]

embeddings = model.encode(sentences, normalize_embeddings=True)
similarity = cosine_similarity(embeddings)

print('Cosine similarity matrix:')
df = pd.DataFrame(
    similarity,
    index=[s[:30] for s in sentences],
    columns=[s[:30] for s in sentences]
)
print(df.round(3))

### 4. Confidence Score Distribution

In [ ]:
sample_texts = [
    'I am devastated and broken inside',
    'Today was just another ordinary day',
    'I feel a bit off but cannot explain why',
    'I am absolutely terrified',
    'Everything is wonderful and I am grateful'
]

confidences = []
emotions = []

for text in sample_texts:
    result = classifier(text)[0]
    top = max(result, key=lambda x: x['score'])
    confidences.append(top['score'])
    emotions.append(top['label'])

plt.figure(figsize=(10, 4))
colors = plt.cm.RdYlGn([c for c in confidences])
bars = plt.bar(range(len(sample_texts)), confidences, color=colors)
plt.xticks(
    range(len(sample_texts)),
    [f"{e}\n{t[:20]}" for e, t in zip(emotions, sample_texts)],
    rotation=20,
    ha='right'
)
plt.ylabel('Confidence Score')
plt.title('Emotion Classification Confidence')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150)
plt.show()
print('Chart saved')